# 导入数据

In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

# 创建或连接 SQLite 数据库
DB_PATH = PROJECT_DIR / "data" / "olist.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

print("数据库连接成功：", DB_PATH)

数据库连接成功： /Users/huawu/Desktop/揾工/实习准备/olist-ecommerce-analysis/data/olist.db


In [6]:
RAW_DIR = PROJECT_DIR / "data" / "raw"

for csv_path in RAW_DIR.glob("*.csv"):
    table_name = csv_path.stem

    df = pd.read_csv(csv_path)
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(df):,} rows")

olist_sellers_dataset: 3,095 rows
product_category_name_translation: 71 rows
olist_orders_dataset: 99,441 rows
olist_order_items_dataset: 112,650 rows
olist_customers_dataset: 99,441 rows
olist_geolocation_dataset: 1,000,163 rows
olist_order_payments_dataset: 103,886 rows
olist_order_reviews_dataset: 99,224 rows
olist_products_dataset: 32,951 rows


In [7]:
sql = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(sql, conn)

,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_order_items_dataset
3,olist_order_payments_dataset
4,olist_order_reviews_dataset
5,olist_orders_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


# 平台经营分析

## 月度交易表现

### 统计每月订单量

In [18]:
sql = """
select 
    strftime("%Y-%m", order_purchase_timestamp) as date,
    count(*) as count
from olist_orders_dataset
where order_status = 'delivered'
group by strftime("%Y-%m", order_purchase_timestamp)
"""

pd.read_sql_query(sql, conn)

,date,count
0,2016-09,1
1,2016-10,265
2,2016-12,1
3,2017-01,750
4,2017-02,1653
5,2017-03,2546
6,2017-04,2303
7,2017-05,3546
8,2017-06,3135
9,2017-07,3872


### 统计每月商品销售额

In [19]:
sql = """
select 
    strftime("%Y-%m", o.order_purchase_timestamp) as date,
    sum(p.price) as sales
from olist_orders_dataset as o
left join olist_order_items_dataset as p
on o.order_id = p.order_id
where o.order_status = 'delivered'
group by date
"""

pd.read_sql_query(sql, conn)

,date,sales
0,2016-09,134.97
1,2016-10,40325.11
2,2016-12,10.90
3,2017-01,111798.36
4,2017-02,234223.40
5,2017-03,359198.85
6,2017-04,340669.68
7,2017-05,489338.25
8,2017-06,421923.37
9,2017-07,481604.52


### 计算每月平均订单金额

In [20]:
sql = """
select
    strftime("%Y-%m", o.order_purchase_timestamp) as date,
    round(sum(p.price)/count(o.order_id), 2) as avg_sale
from olist_orders_dataset as o
left join olist_order_items_dataset as p
on o.order_id = p.order_id
where o.order_status = 'delivered'
group by date
"""
pd.read_sql_query(sql, conn)

,date,avg_sale
0,2016-09,44.99
1,2016-10,128.83
2,2016-12,10.90
3,2017-01,122.45
4,2017-02,126.06
5,2017-03,123.99
6,2017-04,132.61
7,2017-05,122.21
8,2017-06,120.93
9,2017-07,109.06


### 比较订单量与销售额的变化趋势

In [22]:
sql = """
select
    strftime("%Y-%m", o.order_purchase_timestamp) as date,
    count(o.order_id) as order_count,
    sum(p.price) as merchandise_sales
from olist_orders_dataset as o
left join olist_order_items_dataset as p
on o.order_id = p.order_id
where o.order_status = 'delivered'
group by date
"""
pd.read_sql_query(sql, conn)

,date,order_count,merchandise_sales
0,2016-09,3,134.97
1,2016-10,313,40325.11
2,2016-12,1,10.90
3,2017-01,913,111798.36
4,2017-02,1858,234223.40
5,2017-03,2897,359198.85
6,2017-04,2569,340669.68
7,2017-05,4004,489338.25
8,2017-06,3489,421923.37
9,2017-07,4416,481604.52


In [27]:
sql = """
with monthly as (
    select
        strftime("%Y-%m", o.order_purchase_timestamp) as date,
        count(o.order_id) as order_count,
        sum(p.price) as mechandise_sales
    from olist_orders_dataset as o
    left join olist_order_items_dataset as p
    on o.order_id = p.order_id
    where o.order_status = 'delivered'
    group by date
),
previous_month as (
    select
        date,
        order_count,
        mechandise_sales,
        lag(order_count) over (
            order by date
        ) as prev_order_count,
        lag(mechandise_sales) over (
            order by date
        ) as prev_mechandise_sales
    from monthly
)
select
    date,
    order_count,
    concat(
        round(
            100*(order_count - prev_order_count)/nullif(prev_order_count, 0),
            2
        ),
        '%'
    ) as order_growth_pct,
    mechandise_sales,
    concat(
        round(
                100*(mechandise_sales - prev_mechandise_sales)/nullif(prev_mechandise_sales, 0),
                2
            ),
        '%'
    ) as sales_growth_pct
from previous_month
"""

pd.read_sql_query(sql, conn)

,date,order_count,order_growth_pct,mechandise_sales,sales_growth_pct
0,2016-09,3,%,134.97,%
1,2016-10,313,10333.0%,40325.11,29777.09%
2,2016-12,1,-99.0%,10.90,-99.97%
3,2017-01,913,91200.0%,111798.36,1025573.03%
4,2017-02,1858,103.0%,234223.40,109.51%
5,2017-03,2897,55.0%,359198.85,53.36%
6,2017-04,2569,-11.0%,340669.68,-5.16%
7,2017-05,4004,55.0%,489338.25,43.64%
8,2017-06,3489,-12.0%,421923.37,-13.78%
9,2017-07,4416,26.0%,481604.52,14.15%


## 商品品类结构

### 按商品品类统计订单量

In [30]:
sql = """
    select
        n.product_category_name_english as category,
        count(o.order_id) as order_count
    from olist_products_dataset as p
    left join olist_order_items_dataset as o
    on p.product_id = o.product_id
    left join product_category_name_translation as n
    on n.product_category_name = p.product_category_name
    group by category
    order by order_count desc
"""

pd.read_sql_query(sql, conn)

,category,order_count
0,bed_bath_table,11115
1,health_beauty,9670
2,sports_leisure,8641
3,furniture_decor,8334
4,computers_accessories,7827
...,...,...
67,arts_and_craftmanship,24
68,la_cuisine,14
69,cds_dvds_musicals,14
70,fashion_childrens_clothes,8


### 统计各品类商品销售额及平均商品价格

In [33]:
sql = """
select
    n.product_category_name_english as category,
    sum(o.price) as sales,
    round(sum(o.price)/count(o.order_id), 2) as avg_price
from olist_order_items_dataset as o
left join olist_products_dataset as p
on o.product_id = p.product_id
left join product_category_name_translation as n
on n.product_category_name = p.product_category_name
group by category
order by sales desc
"""
pd.read_sql_query(sql, conn)

,category,sales,avg_price
0,health_beauty,1258681.34,130.16
1,watches_gifts,1205005.68,201.14
2,bed_bath_table,1036988.68,93.30
3,sports_leisure,988048.97,114.34
4,computers_accessories,911954.32,116.51
...,...,...,...
67,flowers,1110.04,33.64
68,home_comfort_2,760.27,25.34
69,cds_dvds_musicals,730.00,52.14
70,fashion_childrens_clothes,569.85,71.23


### 计算各品类销售额占平台总销售额的比例

In [38]:
sql = """
with category_sales as (
    select
        n.product_category_name_english as category,
        sum(o.price) as sales
    from olist_order_items_dataset as o
    left join olist_products_dataset as p
    on o.product_id = p.product_id
    left join product_category_name_translation as n
    on n.product_category_name = p.product_category_name
    group by category
    order by sales desc
)  
select
    category,
    sales,
    concat(
        round(
            100*sales / sum(sales) over (),
            2
        ),
        '%'
    ) as sales_pct
from category_sales
"""
pd.read_sql_query(sql, conn)

,category,sales,sales_pct
0,health_beauty,1258681.34,9.26%
1,watches_gifts,1205005.68,8.87%
2,bed_bath_table,1036988.68,7.63%
3,sports_leisure,988048.97,7.27%
4,computers_accessories,911954.32,6.71%
...,...,...,...
67,flowers,1110.04,0.01%
68,home_comfort_2,760.27,0.01%
69,cds_dvds_musicals,730.00,0.01%
70,fashion_childrens_clothes,569.85,0.0%


### 对品类进行销售额排名

In [40]:
sql = """
select
    n.product_category_name_english as category,
    sum(o.price) as sales,
    rank() over (order by sum(o.price) desc) as sales_rank
from olist_order_items_dataset as o
left join olist_products_dataset as p
on o.product_id = p.product_id
left join product_category_name_translation as n
on n.product_category_name = p.product_category_name
group by category
order by sales desc
"""
pd.read_sql_query(sql, conn)

,category,sales,sales_rank
0,health_beauty,1258681.34,1
1,watches_gifts,1205005.68,2
2,bed_bath_table,1036988.68,3
3,sports_leisure,988048.97,4
4,computers_accessories,911954.32,5
...,...,...,...
67,flowers,1110.04,68
68,home_comfort_2,760.27,69
69,cds_dvds_musicals,730.00,70
70,fashion_childrens_clothes,569.85,71


## 订单履约表现

### 统计整体订单延迟率

In [48]:
sql = """
select
    concat(
        round(
            100*avg(
                case
                    when order_delivered_customer_date > order_estimated_delivery_date then 1
                    else 0
                    end
            ) ,
            2
        ),
        '%'
    ) as delay_pct
from olist_orders_dataset
where order_delivered_customer_date is not null 
    and order_estimated_delivery_date is not null
    and order_status = 'delivered'
"""
pd.read_sql_query(sql, conn)

,delay_pct
0,8.11%


### 计算订单实际配送时长

In [49]:
sql = """
select 
    order_id,
    round(
        julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp),
        2
    ) as delivery_days
from olist_orders_dataset
where order_delivered_customer_date is not null 
    and order_estimated_delivery_date is not null
    and order_status = 'delivered'
"""

pd.read_sql_query(sql, conn)

,order_id,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.44
1,53cdb2fc8bc7dce0b6741e2150273451,13.78
2,47770eb9100c2d0c44946d9cf07ec65d,9.39
3,949d5b44dbf5de918fe9c16f97b45f8a,13.21
4,ad21c59c0840e6cb83a9ceb5573f8159,2.87
...,...,...
96465,9c5dedf39a927c1b2549525ed64a053c,8.22
96466,63943bddc261676b46f01ca7ac2f7bd8,22.19
96467,83c1379a015df1e13d02aae0204711ab,24.86
96468,11c177c8e97725db2631073c19f07b62,17.09


### 按月份比较订单延迟率

In [51]:
sql = """
select 
    strftime('%Y-%m', order_purchase_timestamp) as date,
    concat(
        round(
            100*avg(
                case
                    when order_delivered_customer_date > order_estimated_delivery_date then 1
                    else 0
                    end    
            ),
            2
        ),
        '%'
    ) as delay_pct
from olist_orders_dataset
where order_delivered_customer_date is not null 
    and order_estimated_delivery_date is not null
    and order_status = 'delivered'
group by date
"""

pd.read_sql_query(sql, conn)

,date,delay_pct
0,2016-09,100.0%
1,2016-10,1.13%
2,2016-12,0.0%
3,2017-01,3.07%
4,2017-02,3.21%
5,2017-03,5.58%
6,2017-04,7.86%
7,2017-05,3.61%
8,2017-06,3.86%
9,2017-07,3.43%


### 按州或地区比较订单延迟率

In [55]:
sql = """
select
    c.customer_state as state,
    concat(
            round(
                100*avg(
                    case
                        when o.order_delivered_customer_date > o.order_estimated_delivery_date then 1
                        else 0
                        end    
                ),
                2
            ),
            '%'
        ) as delay_pct
from olist_orders_dataset as o
left join olist_customers_dataset as c
on o.customer_id = c.customer_id
where order_delivered_customer_date is not null 
    and order_estimated_delivery_date is not null
    and order_status = 'delivered'
group by state
order by avg(
                    case
                        when o.order_delivered_customer_date > o.order_estimated_delivery_date then 1
                        else 0
                        end    
                ) desc 
"""
pd.read_sql_query(sql, conn)

,state,delay_pct
0,AL,23.93%
1,MA,19.67%
2,PI,15.97%
3,CE,15.32%
4,SE,15.22%
5,BA,14.04%
6,RJ,13.47%
7,TO,12.77%
8,PA,12.37%
9,ES,12.23%


## 卖家经营表现

### 统计各卖家的订单量和商品销售额

In [59]:
sql = """
select 
    distinct s.seller_id,
    sum(o.price) as sales
from olist_sellers_dataset as s
left join olist_order_items_dataset as o
on s.seller_id = o.seller_id
group by s.seller_id
order by sales desc
"""

pd.read_sql_query(sql, conn)

,seller_id,sales
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00
3091,702835e4b785b67a084280efca355756,7.60
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90
3093,77128dec4bec4878c37ab7d6169d6f26,6.50


### 统计各卖家的平均用户评分

In [63]:
sql = """
select
    distinct s.seller_id,
    round(avg(r.review_score), 2) as avg_score
from olist_sellers_dataset as s
left join olist_order_items_dataset as o
on s.seller_id = o.seller_id
left join olist_order_reviews_dataset as r
on o.order_id = r.order_id
where r.review_score is not null
group by s.seller_id
order by avg_score desc
"""
pd.read_sql_query(sql, conn)

,seller_id,avg_score
0,ff314fa6033cc68ec451c47aee2d6ba4,5.0
1,feb793c88d836c3a75efa4b5a3465d70,5.0
2,fe19dce63ae80346207c6c55713d1023,5.0
3,fdf736c18c589ed030e058312203e1b2,5.0
4,fd435faa3c0422b60440ea3480d0e77c,5.0
...,...,...
3085,154bdf805377afea75a3bd158e9eab10,1.0
3086,1444c08e64d55fb3c25f0f09c07ffcf2,1.0
3087,0aa124728afc1131dff5655f4c6f487b,1.0
3088,010da0602d7774602cd1b3f5fb7b709e,1.0


### 计算各卖家的订单延迟率

In [67]:
sql = """
select
    s.seller_id,
    concat(
        round(
            100*avg(
                case
                    when o.order_delivered_customer_date > o.order_estimated_delivery_date then 1
                    else 0
                    end
            ),
            2
        ),
        '%'
    ) as delay_pct
from olist_orders_dataset as o
left join olist_order_items_dataset as i
on o.order_id = i.order_id
left join olist_sellers_dataset as s
on i.seller_id = s.seller_id
where o.order_delivered_customer_date is not null
    and o.order_estimated_delivery_date is not null
group by s.seller_id
order by avg(
                case
                    when o.order_delivered_customer_date > o.order_estimated_delivery_date then 1
                    else 0
                    end
            ) desc
"""

pd.read_sql_query(sql, conn)

,seller_id,delay_pct
0,f5fea3ffed6c2e889bab72705557c63a,100.0%
1,f524ad65d7e0f1daab730ef2d2e86196,100.0%
2,efb628aa07c8ca93d7db0d85b91a4f5a,100.0%
3,edf3fabebcc20f7463cc9c53da932ea8,100.0%
4,ebe3dabc4955c9b0d695e7bda38bf5c3,100.0%
...,...,...
2965,010543a62bd80aa422851e79a3bc7540,0.0%
2966,00d8b143d12632bad99c0ad66ad52825,0.0%
2967,00ab3eff1b5192e5f1a63bcecfee11c8,0.0%
2968,003554e2dce176b5555353e4f3555ac8,0.0%


### 筛选交易规模较大但用户反馈较差的卖家

In [73]:
sql = """
select
    s.seller_id,
    sum(o.price) as sales,
    count(o.order_id) as orders,
    avg(r.review_score) as avg_score
from olist_order_items_dataset as o
left join olist_sellers_dataset as s
on o.seller_id = s.seller_id
left join olist_order_reviews_dataset as r
on o.order_id = r.order_id
where r.review_score is not null
group by s.seller_id
having avg_score < 3.5 and sales > 10000
order by
    sales desc,
    orders desc,
    avg_score asc
"""

pd.read_sql_query(sql, conn)

,seller_id,sales,orders,avg_score
0,7c67e1448b00f6e969d365cea6b010ab,188017.85,1367,3.348208
1,2eb70248d66e0e3ef83659f71b244378,41307.84,209,2.712919
2,712e6ed8aa4aa1fa65dab41fed5737e4,39318.00,87,3.459770
3,88460e8ebdecbfecb5f9601833981930,32334.40,310,3.351613
4,b1b3948701c5c72445495bd161b83a4c,24699.19,18,1.722222
5,b37c4c02bda3161a7546a4e6d222d5b2,24075.00,15,1.400000
6,710e3548e02bc1d2831dfc4f1b5b14d4,23869.12,141,3.397163
7,897060da8b9a21f655304d50fd935913,23537.17,430,3.395349
8,8444e55c1f13cd5c179851e5ca5ebd00,22873.21,109,3.073394
9,40db9e9aa57f7bb151bcda6b0f9bdbb7,21380.00,11,2.363636


# 用户消费与体验分析

## 用户消费行为

### 统计每位用户的购买订单数

In [78]:
sql = """
select
    c.customer_unique_id,
    count(o.order_id) as orders
from olist_orders_dataset as o
left join olist_customers_dataset as c
on o.customer_id = c.customer_id
where o.order_status = 'delivered'
group by c.customer_unique_id
order by orders desc
"""
pd.read_sql_query(sql, conn)

,customer_unique_id,orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,15
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
...,...,...
93353,0004aac84e0df4da2b147fca70cf8255,1
93354,0000f6ccb0745a6a4b88665a16c9f078,1
93355,0000f46a3911fa3c0805444483337064,1
93356,0000b849f77a49e4a4ce2b2a4ca5be3f,1


### 统计每位用户的累计消费金额和平均订单金额

In [82]:
sql = """
select
    c.customer_unique_id,
    sum(i.price) as consume,
    sum(i.price)/count(o.order_id) as avg_price
from olist_orders_dataset as o
left join olist_customers_dataset as c
on o.customer_id = c.customer_id
left join olist_order_items_dataset as i
on o.order_id = i.order_id
where o.order_status = 'delivered'
group by c.customer_unique_id
order by 
    consume desc,
    avg_price desc
"""
pd.read_sql_query(sql, conn)

,customer_unique_id,consume,avg_price
0,0a0a92112bd4c708ca5fde585afaa872,13440.00,1680.00
1,da122df9eeddfedc1dc1f5349a1a690c,7388.00,3694.00
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.00,1790.00
3,dc4802a71eae9be1dd28f5d788ceb526,6735.00,6735.00
4,459bef486812aa25204be022145caa62,6729.00,6729.00
...,...,...,...
93353,cf3839da0d9492ad151690b65f45d800,2.99,2.99
93354,bd06ce0e06ad77a7f681f1a4960a3cc6,2.29,2.29
93355,317cfc692e3f86c45c95697c61c853a6,2.20,2.20
93356,b38211bd797f4fdd81a98b9d1754b606,0.85,0.85


### 分析用户购买订单数的分布

In [83]:
sql = """
with customer_order_counts as (
    select
        c.customer_unique_id,
        count(o.order_id) as order_count
    from olist_orders_dataset as o
    left join olist_customers_dataset as c
    on o.customer_id = c.customer_id
    where o.order_status = 'delivered'
    group by c.customer_unique_id  
)
select
    order_count,
    count(customer_unique_id) as user_counts
from customer_order_counts
group by order_count
order by order_count
"""
pd.read_sql_query(sql, conn)

,order_count,user_counts
0,1,90557
1,2,2573
2,3,181
3,4,28
4,5,9
5,6,5
6,7,3
7,9,1
8,15,1


### 统计不同消费频次用户的数量及占比

In [91]:
sql = """
with customer_order_counts as (
    select
        c.customer_unique_id,
        count(o.order_id) as order_count
    from olist_orders_dataset as o
    left join olist_customers_dataset as c
    on o.customer_id = c.customer_id
    where o.order_status = 'delivered'
    group by c.customer_unique_id  
),
freq as (
    select
        order_count,
        count(customer_unique_id) as user_counts,
        (
            case
                when order_count = 1 then '一次购买用户'
                when order_count between 2 and 3 then '低频购买用户'
                when order_count >= 4 then '高频购买用户'
                end
        ) as freq_type
    from customer_order_counts
    group by order_count
    order by order_count
)
select *
from freq
group by freq_type
"""
pd.read_sql_query(sql, conn)

,order_count,user_counts,freq_type
0,1,90557,一次购买用户
1,2,2573,低频购买用户
2,4,28,高频购买用户


## 用户复购情况

### 统计至少购买两笔订单的用户数量及占比

In [96]:
sql = """
select
    c.customer_unique_id,
    count(o.order_id) as order_count
from olist_customers_dataset as c
left join olist_orders_dataset as o
on c.customer_id = o.customer_id
group by c.customer_unique_id
having order_count > 1
"""
pd.read_sql_query(sql, conn)

,customer_unique_id,order_count
0,00172711b30d52eea8b313a7f2cced02,2
1,004288347e5e88a27ded2bb23747066c,2
2,004b45ec5c64187465168251cd1c9c2f,2
3,0058f300f57d7b93c477a131a59b36c3,2
4,00a39521eb40f7012db50455bf083460,2
...,...,...
2992,ff36be26206fffe1eb37afd54c70e18b,3
2993,ff44401d0d8f5b9c54a47374eb48c1b8,2
2994,ff8892f7c26aa0446da53d01b18df463,2
2995,ff922bdd6bafcdf99cb90d7f39cea5b3,3


In [101]:
sql = """
with order_count as (
    select
        c.customer_unique_id,
        count(o.order_id) as order_count
    from olist_customers_dataset as c
    left join olist_orders_dataset as o
    on c.customer_id = o.customer_id
    group by c.customer_unique_id
)
select
    sum(
        case
            when order_count > 1 then 1
            else 0
            end
    ) as again_count,
    round(
        avg(
            case
                when order_count > 1 then 1
                else 0
                end
        ),
        2
    ) as again_pct
from order_count
"""
pd.read_sql_query(sql, conn)

,again_count,again_pct
0,2997,0.03


### 计算用户首次购买时间

In [103]:
sql = """
select
    c.customer_unique_id,
    min(o.order_purchase_timestamp) as first_purchase_date
from olist_orders_dataset as o
left join olist_customers_dataset as c
on o.customer_id = c.customer_id
where o.order_status = 'delivered'
group by c.customer_unique_id
"""
pd.read_sql_query(sql, conn)

,customer_unique_id,first_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42
...,...,...
93353,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:36
93354,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:56
93355,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:16
93356,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:41


### 按首次购买月份划分用户群体

In [112]:
sql = """
select
    c.customer_unique_id,
    min(o.order_purchase_timestamp) as first_purchase_date
from olist_orders_dataset as o
left join olist_customers_dataset as c
on o.customer_id = c.customer_id
where o.order_status = 'delivered'
group by c.customer_unique_id
order by first_purchase_date
"""
pd.read_sql_query(sql, conn)

,customer_unique_id,first_purchase_date
0,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38
1,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50
2,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50
3,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36
4,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03
...,...,...
93353,7a22d14aa3c3599238509ddca4b93b01,2018-08-29 12:25:59
93354,5c58de6fb80e93396e2f35642666b693,2018-08-29 14:18:23
93355,7febafa06d9d8f232a900a2937f04338,2018-08-29 14:18:28
93356,b701bebbdf478f5500348f03aff62121,2018-08-29 14:52:00


In [111]:
sql = """
with time_t as (
    select
        c.customer_unique_id,
        min(o.order_purchase_timestamp) as first_purchase_date
    from olist_orders_dataset as o
    left join olist_customers_dataset as c
    on o.customer_id = c.customer_id
    where o.order_status = 'delivered'
    group by c.customer_unique_id
)
select
    strftime('%Y-%m', first_purchase_date) as date,
    count(*) as customers
from time_t
group by date
"""
pd.read_sql_query(sql, conn)

,date,customers
0,2016-09,1
1,2016-10,262
2,2016-12,1
3,2017-01,717
4,2017-02,1628
5,2017-03,2503
6,2017-04,2256
7,2017-05,3451
8,2017-06,3037
9,2017-07,3752


### 分析不同月份首次购买的用户在后续月份是否再次购买

In [126]:
sql = """
with first_purchase as (
    select
        c.customer_unique_id,
        min(o.order_purchase_timestamp) as first_purchase_date
    from olist_orders_dataset as o
    left join olist_customers_dataset as c
    on o.customer_id = c.customer_id
    where o.order_status = 'delivered'
    group by c.customer_unique_id
),
customer_month as (
    select
        c.customer_unique_id,
        strftime('%Y-%m', o.order_purchase_timestamp) as purchase_month
    from olist_orders_dataset as o
    left join olist_customers_dataset as c
    on o.customer_id = c.customer_id
    where o.order_status = 'delivered'
),
customer_activity as (
    select
        cm.customer_unique_id,
        strftime('%Y-%m', fp.first_purchase_date) as first_purchase_month,
        cm.purchase_month
    from customer_month as cm
    left join first_purchase as fp
    on cm.customer_unique_id = fp.customer_unique_id
),
again_analysis as (
    select 
        first_purchase_month,
        (
            case
                when purchase_month > first_purchase_month then 'again'
                else 'first'
                end
        ) as again
    from customer_activity
    order by first_purchase_month
)
select
    first_purchase_month,
    sum(
        case
            when again = 'again' then 1
            else 0
            end
    ) as again_count,
    concat(
        round(
            100*avg(
                case
                    when again = 'again' then 1
                    else 0 
                    end
            ),
            2
        ),
        '%'
    ) as again_pct
from again_analysis
group by first_purchase_month
"""
pd.read_sql_query(sql, conn)

,first_purchase_month,again_count,again_pct
0,2016-09,0,0.0%
1,2016-10,10,3.64%
2,2016-12,1,50.0%
3,2017-01,34,4.34%
4,2017-02,50,2.94%
5,2017-03,103,3.9%
6,2017-04,83,3.51%
7,2017-05,149,4.06%
8,2017-06,134,4.15%
9,2017-07,134,3.39%


## 用户评分分布

### 统计各评分等级的评价数量及占比

In [133]:
sql = """
select
    review_score,
    count(*) as count,
    concat(
        round(
            100*1.0*count(*)/(sum(count(*)) over ()),
            2
        ),
        '%' 
    ) as percentage
from olist_order_reviews_dataset
group by review_score
"""
pd.read_sql_query(sql, conn)

,review_score,count,percentage
0,1,11424,11.51%
1,2,3151,3.18%
2,3,8179,8.24%
3,4,19142,19.29%
4,5,57328,57.78%


### 计算平台平均评分和低分评价率（1–2 分）

In [ ]:
sql = """
select
    round(
        1.0*sum(review_score)/count(*),
        2
    ) as avg_score,
    concat(
        round(
            100*avg(
                case
                    when review_score between 1 and 2 then 1
                    else 0 
                    end
            ),
            2
        ),
        '%'
    ) as low_score_pct
from olist_order_reviews_dataset
"""
pd.read_sql_query(sql, conn)

,avg_score,low_score_pct
0,4.09,14.69%


### 按商品品类比较低分评价率

In [143]:
sql = """
select
    n.product_category_name_english,
    concat(
            round(
                100*avg(
                    case
                        when r.review_score between 1 and 2 then 1
                        else 0 
                        end
                ),
                2
            ),
            '%'
        ) as low_score_pct
from olist_order_items_dataset as i
left join olist_products_dataset as p
on i.product_id = p.product_id
left join product_category_name_translation as n
on n.product_category_name = p.product_category_name
left join olist_order_reviews_dataset as r
on i.order_id = r.order_id
where p.product_category_name is not null and n.product_category_name_english is not null
group by n.product_category_name_english
order by avg(
                    case
                        when r.review_score between 1 and 2 then 1
                        else 0 
                        end
                ) desc
"""
pd.read_sql_query(sql, conn)

,product_category_name_english,low_score_pct
0,security_and_services,50.0%
1,fashion_male_clothing,28.03%
2,office_furniture,25.87%
3,diapers_and_hygiene,25.64%
4,fixed_telephony,25.28%
...,...,...
66,books_general_interest,8.5%
67,food_drink,7.86%
68,costruction_tools_tools,7.77%
69,flowers,6.06%


### 按卖家或地区比较低分评价率

In [146]:
sql = """
select
    s.seller_id,
    concat(
                round(
                    100*avg(
                        case
                            when r.review_score between 1 and 2 then 1
                            else 0 
                            end
                    ),
                    2
                ),
                '%'
            ) as low_score_pct
from olist_sellers_dataset as s
left join olist_order_items_dataset as i
on s.seller_id = i.seller_id
left join olist_order_reviews_dataset as r
on i.order_id = r.order_id
group by s.seller_id
order by avg(
                case
                    when r.review_score between 1 and 2 then 1
                    else 0 
                    end
                ) desc
"""
pd.read_sql_query(sql, conn)

,seller_id,low_score_pct
0,ffcfefa19b08742c5d315f2791395ee5,100.0%
1,fec6912baad85d41729669edd6b4d3b8,100.0%
2,fc99f99385ae90f66f01733856fb88c9,100.0%
3,fc6295add6f51a0936407ead70c1001d,100.0%
4,fa7ad8ae9bcb2c4493fdf38e96b1d74f,100.0%
...,...,...
3090,011b0eaba87386a2ae96a7d32bb531d1,0.0%
3091,010543a62bd80aa422851e79a3bc7540,0.0%
3092,00d8b143d12632bad99c0ad66ad52825,0.0%
3093,00ab3eff1b5192e5f1a63bcecfee11c8,0.0%


In [147]:
sql = """
select
    s.seller_city,
    concat(
                round(
                    100*avg(
                        case
                            when r.review_score between 1 and 2 then 1
                            else 0 
                            end
                    ),
                    2
                ),
                '%'
            ) as low_score_pct
from olist_sellers_dataset as s
left join olist_order_items_dataset as i
on s.seller_id = i.seller_id
left join olist_order_reviews_dataset as r
on i.order_id = r.order_id
group by s.seller_city
order by avg(
                case
                    when r.review_score between 1 and 2 then 1
                    else 0 
                    end
                ) desc
"""
pd.read_sql_query(sql, conn)

,seller_city,low_score_pct
0,vera cruz,100.0%
1,soledade,100.0%
2,santo antonio da patrulha,100.0%
3,rio branco,100.0%
4,portoferreira,100.0%
...,...,...
606,angra dos reis,0.0%
607,ampere,0.0%
608,afonso claudio,0.0%
609,abadia de goias,0.0%


In [148]:
sql = """
select
    s.seller_state,
    concat(
                round(
                    100*avg(
                        case
                            when r.review_score between 1 and 2 then 1
                            else 0 
                            end
                    ),
                    2
                ),
                '%'
            ) as low_score_pct
from olist_sellers_dataset as s
left join olist_order_items_dataset as i
on s.seller_id = i.seller_id
left join olist_order_reviews_dataset as r
on i.order_id = r.order_id
group by s.seller_state
order by avg(
                case
                    when r.review_score between 1 and 2 then 1
                    else 0 
                    end
                ) desc
"""
pd.read_sql_query(sql, conn)

,seller_state,low_score_pct
0,AC,100.0%
1,AM,66.67%
2,RO,28.57%
3,PB,23.68%
4,SE,20.0%
5,PI,16.67%
6,SP,16.58%
7,ES,16.13%
8,MA,16.01%
9,DF,15.98%


## 用户体验影响因素探索

### 比较高分订单与低分订单的配送时长

In [150]:
sql = """
select
    (
        case
            when r.review_score between 1 and 2 then '低分订单'
            when r.review_score between 4 and 5 then '高分订单'
        end
    ) as order_type,
    count(distinct o.order_id) as order_count,
    round(
        avg(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)),
        2
    ) as avg_delivered_days
from olist_orders_dataset as o
left join olist_order_reviews_dataset as r
on o.order_id = r.order_id
where o.order_status = 'delivered'
    and o.order_delivered_customer_date is not null
    and o.order_purchase_timestamp is not null
    and r.review_score in (1, 2, 4, 5)
group by order_type
"""
pd.read_sql_query(sql, conn)

,order_type,order_count,avg_delivered_days
0,低分订单,12310,20.20
1,高分订单,75700,11.09


### 比较延迟订单与准时订单的评分差异

In [153]:
sql = """
select
    (
        case
            when o.order_delivered_customer_date > o.order_estimated_delivery_date then 'delay'
            when o.order_delivered_customer_date < o.order_estimated_delivery_date then 'on time'
        end
    ) as order_type,
    round(
        avg(r.review_score),
        2
    ) as avg_score
from olist_orders_dataset as o
left join olist_order_reviews_dataset as r
on o.order_id = r.order_id
where o.order_status = 'delivered'
    and o.order_delivered_customer_date is not null
    and o.order_estimated_delivery_date is not null
    and r.review_score is not null
group by order_type
"""
pd.read_sql_query(sql, conn)

,order_type,avg_score
0,delay,2.57
1,on time,4.29


### 比较不同商品金额、运费区间的用户评分

In [157]:
sql = """
with groups as (
    select
        ntile(10) over(
            order by price
        ) as price_group,
        order_id
    from olist_order_items_dataset
)
select
    g.price_group,
    round(
        avg(r.review_score),
        4
    ) as avg_score
from groups as g
left join olist_order_reviews_dataset as r
on g.order_id = r.order_id
group by price_group
"""
pd.read_sql_query(sql, conn)

,price_group,avg_score
0,1,3.9886
1,2,4.0718
2,3,4.0368
3,4,4.0227
4,5,4.0365
5,6,3.9778
6,7,4.0837
7,8,4.0357
8,9,4.0472
9,10,4.0239


In [158]:
sql = """
with groups as (
    select
        ntile(10) over(
            order by freight_value
        ) as freight_group,
        order_id
    from olist_order_items_dataset
)
select
    g.freight_group,
    round(
        avg(r.review_score),
        4
    ) as avg_score
from groups as g
left join olist_order_reviews_dataset as r
on g.order_id = r.order_id
group by freight_group
"""
pd.read_sql_query(sql, conn)

,freight_group,avg_score
0,1,4.0825
1,2,4.1174
2,3,4.1682
3,4,4.0326
4,5,4.0290
5,6,3.9782
6,7,4.0516
7,8,3.9842
8,9,3.9689
9,10,3.9114
